# Redis Index Sliding-Window

Indexierung der PDF-Dokumente im Redis Vector-Store für Retrieval mittels Sliding Window und Overlapping.

**Voraussetzung**

Environment: dski

Emebdding-Modell mittels vLLM über Rest-API zur Verfügung stellen und Redis-Server für Indexierung der Embedding-Vektoren starten (siehe setup.md).

**Indexierung**

Die Indexierung wird für die 245 Dokumente durchgeführt, auf welche sich die 397 gewählten Fragen aus dem Datensatz beziehen.

Alle Dokumente werden für das Retrieval mit dem Verfahren Sliding-Window unter Verwendung von Fenstern mit 512 und 1024 Zeichen sowie Überlappungen von jeweils 0%, 50% und 100% in Abschnitte unterteilt. Für die erstellten Abschnitte werden durch das Embedding-Modell Embedding-Vektoren mit der Dimension 2048 erzeugt und diese im Vector-Store indexiert.

## Setup

### Pakete laden

In [1]:
import config

#import os
#import json
import math
import time
import hashlib

import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq

from openai import OpenAI

import redis

from redis.commands.search.field import NumericField, TagField, TextField, VectorField
from redis.commands.search.index_definition import IndexDefinition, IndexType
from redis.commands.search.query import Query

### Variablen+Funktionen

In [2]:
# Redis Client
redis_client = redis.Redis(host=config.redis_client['host'], port=config.redis_client['port'], password=config.redis_client['password'], decode_responses=True)

# OpenAI Client
oai_client_embedding = OpenAI(
    # defaults to os.environ.get('OPENAI_API_KEY')
    api_key=config.openai_client['api_key'],
    base_url=config.openai_client['embedding']['base_url'],
)

### Segmentierung - Sliding Window

Funktion zur Unterteilung des Text eines PDF-Dokument in Abschnitte mit Sliding Window und Überlappungen auf Zeichenebene.

In [3]:
def process_document(file_hash, file, pipeline, content, block_size = 512, block_size_overlap = 0.0, embedding_dimension=348):
    # Anzahl Bläcke berechnen
    blocks = math.ceil(len(content) / block_size)
    blocks_content = []
    #if len(content) < block_size:
    for i in range(blocks):
        # Ende des Blocks
        block_end = (i + 1) * block_size
        # Wenn Ende größer als maximale Zeichenlänge
        if block_end > len(content):
            # Ende auf letztes Zeichen setzen
            block_end = len(content)

        # Inhalt des Blocks
        block_content = ''

        # Overlapping linke Seite des Blocks
        if block_size_overlap > 0.0 and i > 0:
            block_content+= content[i * block_size - round(block_size * block_size_overlap): i * block_size].strip() + ' '

        # Block Inhalt hinzufügen
        block_content+= content[i * block_size: block_end].strip()

        # Overlapping rechte Seite des Blocks
        if block_size_overlap > 0.0 and i + 1 < blocks:
            block_end_next = block_end + round(block_size * block_size_overlap)
            if block_end_next > len(content):
                block_end_next = len(content)
            block_content+= ' ' + content[block_end: block_end_next].strip()

        # Block zu Liste hinzufügen
        blocks_content.append(block_content)

        # Block zu Store hinzufügen
        pipeline.json().set(
            f'document:{file_hash}:{i}', 
            '$', 
            {'document': file, 'content': block_content}
        )

    # Embeddings für Blöcke erzeugen
    embeddings = oai_client_embedding.embeddings.create(
        input=blocks_content,
        model=config.models['embedding'][0]
    )

    # Embedding des Blocks zu Einträgen hinzufügen
    # Über Blöcke iterieren
    for i in range(blocks):
        pipeline.json().set(
            f'document:{file_hash}:{i}', 
            '$.content_embeddings', embeddings.data[i].embedding[:embedding_dimension]
        )

    res = pipeline.execute()

### Daten laden

Beispiel-Fragen, Referenz-Antworten und PDF-URLs aus Datensatz laden.

In [4]:
# Answers
answers = pd.read_json(f'{config.dataset['path']}answers.json', orient='index', dtype_backend='numpy_nullable').rename(columns={0: 'answer'})
# Queries
queries = pd.read_json(f'{config.dataset['path']}queries.json', orient='index', dtype_backend='numpy_nullable')
# Relations
qrels = pd.read_json(f'{config.dataset['path']}qrels.json', orient='index', dtype_backend='numpy_nullable')

# PDF URLs
#pdf_urls = pd.read_json(f'{config.dataset['path']}pdf_urls.json', orient='index', dtype_backend='numpy_nullable').rename(columns={0: 'pdf_urls'})

pdf_urls_clean = pd.read_json(f'{config.dataset['path']}pdf_urls_clean.json', orient='index', dtype_backend='numpy_nullable')#.rename(columns={0: 'pdf_urls'})
pdf_urls = pdf_urls_clean.loc[pdf_urls_clean.values]

### Fragen selektieren

Fragen mit Quelle "Text" und "Text-Tabelle" und mindestens acht Wörtern in der Referenz-Antwort (Fragen mit kurzen Antworten wie "yes" ausschließen) für Ausführung selektieren

In [6]:
# Fragen und Antworten über Index joinen
queries_answers = queries.merge(answers, how='inner', left_index=True, right_index=True)

# Anzahl Wörter der Fragen 
queries_answers['answer_replace'] = queries_answers['answer'].str.replace(r'\p{P}+', '', regex=True).str.replace(r'[\n\t\s]+', ' ', regex=True)
queries_answers['answer_word_count'] = queries_answers['answer_replace'].str.split(' ').str.len()

# Fragen mit source=text|text-table und answer_word_count>=8
queries_text_text_table = queries_answers.loc[((queries_answers['source'] == 'text') | (queries_answers['source'] == 'text-table')) & (queries_answers['answer_word_count'] >= 8)]

# 1/4 der Fragen selektieren
queries_text_text_table = queries_text_text_table.sample(n=round(queries_text_text_table.shape[0] / 4), axis=0, random_state=42)

In [7]:
# Beziehung zwischen Frage und Dokument über qrels
# qrels/Dokumente für alle Frage wählen
qrels_select = qrels.loc[queries_text_text_table.index]
# qrels ausschließen, für die Dokumente (doc_id) nicht in pdf_urls (index)
qrels_select = qrels_select.drop(qrels_select.loc[~qrels_select['doc_id'].isin(pdf_urls.index)].index)

pdf_urls_select = pdf_urls.loc[qrels_select['doc_id'].unique()]

## Indexierung

### PDF-Dokumente indexieren

Alle PDF-Dokumente, auf welche sich Fragen beziehen, einlesen und abschnittsweise in redis indexieren

In [8]:
# Index löschen
redis_client.flushdb()
#redis_client.keys('document:*')

block_size = 512#512, 1024
block_size_overlap = 1.0#0.0, 0.5, 1.0

# Dimension der Embedding-Vektoren
embedding_dimension = 2048

# Redis Pipeline erstellen
pipeline = redis_client.pipeline()

# Liste für Dauer der Verarbeitung der Dokumente
process_time = []

# Zeitstempel Start
time_start = time.time()

# Über pdf_urls/Dokumente iterieren
#for doc_id in qrels.loc[queries_text_text_table.index, 'doc_id'].unique():
for idx_doc in range(pdf_urls_select.shape[0]):
    # Dokument-ID
    doc_id = pdf_urls_select.iloc[idx_doc].name
    print(f'{idx_doc + 1} {doc_id}')
    
    # Zeitstempel Start
    process_time_start = time.time()
    
    # Hashwert der Dokument-ID für Keys berechnen
    h = hashlib.sha256()
    h.update(doc_id.encode())

    # PDF-Dokument (herunter)laden
    #Dataset.get_file(doc_id, f'{path}pdfs/')
    # PDF-Dokument in Dataframe einlesen
    #df_doc = PDFProcessor.doc_to_dataframe(f'{path}pdfs/{doc_id}.pdf')
    #df_doc = PDFProcessor.process_dataframe(df_doc, doc_id, path)

    # Dataframe mit vorverarbeiteten Daten laden
    df_doc = pq.read_table(f'{config.dataset['path']}pdfs_df/{doc_id}.parquet').to_pandas()
    # Dokument verarbeiten
    process_document(h.hexdigest(), doc_id, pipeline, df_doc['text'].agg('\n\n'.join), block_size, block_size_overlap, embedding_dimension)

    #res = pipeline.execute()

    process_time.append(time.time() - process_time_start)

print(time.time() - time_start)

1 2406.06650v2
2 2412.05495v6
3 2412.15929v3
4 2410.19654v2
5 2409.12776v2
6 2409.14246v2
7 2410.16076v3
8 2401.14085v2
9 2410.24112v2
10 2408.04814v3
11 2412.04468v2
12 2404.17509v2
13 2407.11894v2
14 2412.15322v2
15 2403.03363v6
16 2408.06103v2
17 2408.16245v3
18 2407.11761v3
19 2411.11402v4
20 2410.08287v3
21 2408.13702v3
22 2401.07294v4
23 2405.13422v2
24 2406.17374v2
25 2412.13042v2
26 2409.11339v2
27 2411.14884v3
28 2405.05734v3
29 2411.03309v2
30 2410.23841v2
31 2405.13879v3
32 2411.10728v3
33 2409.18681v6
34 2405.20415v3
35 2408.14507v2
36 2408.17106v2
37 2411.17395v2
38 2410.07168v2
39 2409.04843v2
40 2411.10511v4
41 2408.06050v2
42 2411.07483v2
43 2403.05373v2
44 2408.06274v3
45 2403.04635v2
46 2412.14566v2
47 2407.02507v2
48 2412.21038v2
49 2404.18137v2
50 2409.06994v3
51 2411.16245v2
52 2405.07102v4
53 2405.01155v3
54 2406.16586v3
55 2409.13333v2
56 2412.06947v3
57 2412.03227v2
58 2410.15316v3
59 2402.04429v3
60 2411.02796v2
61 2409.16644v2
62 2410.11034v2
63 2407.17674v2
6

In [18]:
print('process_time')
print(f'mean {np.mean(process_time)}, min {np.min(process_time)}, max {np.max(process_time)}, std {np.std(process_time)}, median {np.median(process_time)}')

# Block-Size 512, Overlapping 0.0
#total time 535.4815154075623, mean 2.1851237579267853, min 0.36169862747192383, max 11.12071943283081, std 1.366293316836012, median 1.90982985496521

# Block-Size 512, Overlapping 0.5
#total time 779.0644047260284, mean 3.17905426901214, min 0.44498372077941895, max 17.397819757461548, std 2.0446716476871245, median 2.7098500728607178

# Block-Size 512, Overlapping 1.0
#total time 1023.2506482601166, mean 3.9635342237900715, min 0.6161384582519531, max 19.734221935272217, std 2.533549876705184, median 3.4784982204437256

# Block-Size 1024, Overlapping 0.0
#total time 408.4481358528137, mean 1.5844767881899464, min 0.27987194061279297, max 8.436211347579956, std 1.0186927899824907, median 1.3817780017852783

# Block-Size 1024, Overlapping 0.5
#total time 585.1750040054321, mean 2.3877527528879594, min 0.38245368003845215, max 12.049721717834473, std 1.5216403127400517, median 2.1276984214782715

# Block-Size 1024, Overlapping 1.0
#total time 800.655375957489, mean 3.267072206613969, min 0.5429883003234863, max 16.381672859191895, std 2.155295300817204, median 2.812443971633911

process_time
mean 3.267072206613969, min 0.5429883003234863, max 16.381672859191895, std 2.155295300817204, median 2.812443971633911


In [9]:
len(redis_client.keys('document:*'))

# block_size=512
#35897

# block_size=1024
#18004

35897

In [46]:
#35897 * 2048 * 2 / 1024 / 1024
#140,22265625

140.22265625

Aus den 245 Dokumenten, auf welche sich die gewählten Fragen beziehen, wurden bei Verwendung einer Fenstergröße von 512 Zeichen 35897 Abschnitte gebildet. Bei einer verwendeten Dimension der Embedding-Vektoren von 2048 und 16 Bit Floats (halbe Präzision, 2 Byte je Wert) ergibt sich ein Speicherbedarf von ca. 140 MB.

### Embeddings erzeugen

Embeddings für alle Abschnitte eines Dokuments erzeugen

In [ ]:
"""time_start = time.time()

keys = sorted(redis_client.keys('document:*'))

content = redis_client.json().mget(keys, '$.content')
content = [f'{item}' for sublist in content for item in sublist]

embeddings = oai_client_embedding.embeddings.create(
    input=content,
    model=config.models['embedding'][0],
    #dimensions=1024
)

pipeline = redis_client.pipeline()

for key, embedding in zip(keys, embeddings.data):
    #embedding_np = np.array(embedding.embedding)
    #scale = 127 / np.max(np.abs(embedding_np))
    pipeline.json().set(key, '$.content_embeddings', embedding.embedding[:embedding_dimension])
    #pipeline.json().set(key, '$.content_embeddings', (embedding_np * scale).astype(np.int8).tolist())

pipeline.execute()

print(time.time() - time_start)"""

### Redis Vector-Index

Redis Vector-Index erstellen

In [10]:
#embedding_dimension = len(responses.data[0].embedding)

schema = (
    TextField('$.document', no_stem=True, sortable=False, as_name='document'),#, no_index=True
    #NumericField('$.page', as_name='page'),
    TextField('$.content', no_stem=True, sortable=False, as_name='content'),
    VectorField(
        '$.content_embeddings',
        'FLAT',#'SVS-VAMANA',
        {
            'TYPE': 'FLOAT16',
            'DIM': embedding_dimension,
            'DISTANCE_METRIC': 'IP', #'COSINE',
        },
        as_name='vector',
    ),
)

definition = IndexDefinition(prefix=['document:'], index_type=IndexType.JSON)
res = redis_client.ft('idx:document').create_index(fields=schema, definition=definition)

In [23]:
info = redis_client.ft('idx:document').info()
print(f'Abschnitte im Index: {info['num_docs']}')
print(f'hash_indexing_failures: {info['hash_indexing_failures']}')

Abschnitte im Index: 35897
hash_indexing_failures: 0


### Index Abfrage

Abfrage der drei ähnlichsten Abschnitte für eine Beispiel-Frage

In [13]:
# Sample index
index = 0

print('query')
print(queries_text_text_table.iloc[index]['query'])
print('reference answer')
print(queries_text_text_table.iloc[index]['answer'])

query                 Is Tae-Yeong Kwak associated with Deep Bio Inc.?
type                                                        extractive
source                                                            text
answer               Yes, Tae-Yeong Kwak is associated with Deep Bi...
answer_replace       Yes TaeYeong Kwak is associated with Deep Bio ...
answer_word_count                                                   12
Name: 98c0423f-da97-4560-9d7e-1f18e594b36e, dtype: object

In [24]:
# Embedding zu Eingabe erzeugen
embeddings = oai_client_embedding.embeddings.create(
    input=[queries_text_text_table.iloc[index]['query']],
    model=config.models['embedding'][0],
)

# Vector-Store Retrieval
query = (
    Query('(*)=>[KNN 3 @vector $query_vector AS vector_score]')
     .sort_by('vector_score', asc=False)
     .return_fields('vector_score', 'id', 'document', 'content')#, 'vector'#, 'page'
     .dialect(2)
)

result = redis_client.ft('idx:document').search(
    query,
    {
        'query_vector': np.array([embeddings.data[0].embedding[:embedding_dimension]], dtype=np.float16).tobytes()
    }
)

result.docs#[0]['vector']

[Document {'id': 'document:cd9893568c93927bb833ae9f0b78c6918e4680d7f67b065cd298685de319584f:66', 'payload': None, 'vector_score': '0.705405056477', 'document': '2406.06650v2', 'content': 'irmed that deep learning models can learn morphological characteristics such as mitotic figures through various visualization methods. Deep learning models generally tended to follow pathologist’s opinion, but they were not completely consistent. Through the results of the above studies, it was found that histological grading could be used to approximate this to some extent, and if we expand it further, we expect to be able to effectively prove recurrence by using common patterns in breast cancer.\n\nDeclarat ions Ethical Approval and Consent to Participate\n\nThis study was reviewed and approved by the Institutional Review Board (IRB) of Korea University Medical Center, Seoul, Korea (IRB No. 2020-GRO-146). This study was performed in accordance with the Declaration of Helsinki. Written informed conse

Beispiel-Frage aus Datensatz

In [25]:
#df_doc = pq.read_table(f'{config.dataset['path']}pdfs_df/{qrels.loc[queries_text_text_table.iloc[index].name, 'doc_id']}.parquet').to_pandas()
#print(df_doc['text'].agg('\n\n'.join))